# Chapter 2 - Lab 6: Investement Recommendation Based on Fundamental Analysis Using Sequential Workflow

In this tutorial, we build a sequential agentic system that retrieves fundamental financial ratios for a given stock from an external API and generates an investment recommendation: **Buy, Hold, or Sell**.


# How do we build a sequential workflow?

The workflow is composed of 3 agents:

- **Agent 1: Fundamental Retrieval Agent**

  Retrieves fundamental ratios from the Financial Modeling Prep (FMP) API (e.g., for Apple) and returns the data as a *structured JSON *output.

- **Agent 2 : Fundamental Analysis Agent**  
  Consumes the financial ratios produced by the first agent and interprets them from a fundamental perspective. Based on these inputs, it evaluates the *strength or weakness of key dimensions such as profitability, valuation, and risk, and provides a short justification for each assessment*. The agent returns its conclusions in a structured JSON format, enabling downstream consumption, validation, or further processing.

- **Agent 3 : Decision Agent**  
  Serves as the final decision-making component of the system. It consumes the structured analysis produced by the second agent and synthesizes this information to issue an investment decision, *Buy, Hold, or Sell*, along with the *reasoning* underlying that conclusion. In addition, the agent highlights key metrics whose improvement or deterioration could lead to a different decision in the future. The final output is returned in a structured JSON format, enabling consistent interpretation and downstream use.


**Disclaimer**: The content presented in this tutorial is provided solely for educational and illustrative purposes. It does not constitute investment advice, financial recommendations, or an offer to buy or sell any securities. Any examples or analyses are hypothetical and should not be relied upon for real-world investment decisions. Readers should conduct their own research and consult a qualified financial professional before making any investment decisions.

# Install Libs

In [ ]:
!pip install openai -q

In [ ]:
%pip install openai-agents -q
%pip install financetoolkit -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.4/284.4 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.7/150.7 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.5/263.5 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 409.7/409.7 kB 13.8 MB/s eta 0:00:00


In [ ]:
import openai
print(openai.__version__)

In [ ]:
from google.colab import userdata
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

import os
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

FINANCIAL_MODELING_PREP_API_KEY = userdata.get('FINANCIAL_MODELING_PREP_API_KEY')

In [ ]:
import nest_asyncio
nest_asyncio.apply()
import pandas as pd
from financetoolkit import Toolkit

from agents import Agent, Runner, WebSearchTool, trace, ItemHelpers, TResponseInputItem, function_tool
from dataclasses import dataclass
from typing import Literal

# Retrieving Fundamental Ratios from FMP (Financial Modeling Prep) API:

Here is the function we'll use to retrieve fundamental ratios.
We'll focus on profitability and valuation ones:

In [ ]:
def get_company_fundamentals(ticker: str) -> dict[str:pd.DataFrame]:
  """ Get the different fundamental ratios for a given ticker. """
  companies = Toolkit(
      [ticker], api_key=FINANCIAL_MODELING_PREP_API_KEY, start_date="2022-01-01"
  )
  ratios = companies.ratios.collect_all_ratios()

  print("ratios", ratios.loc['Return on Assets'])

  # Keep only profitability and valuation ratios
  profitability_ratios_names = ['Return on Equity','Return on Assets','Gross Margin','Operating Margin','Net Profit Margin']
  valuation_ratios_names = ['Price-to-Earnings',  "Price-to-Book", "Price-to-Cash-Flow", "Price-to-Free-Cash-Flow", 'EV-to-EBITDA','EV-to-Sales']
  # You can also use: companies.ratios.collect_profitability_ratios() or companies.ratios.collect_valuation_ratios()


  return {"profitability_ratios":ratios.loc[ratios.index.intersection(profitability_ratios_names)],
   "valuation_ratios":ratios.loc[ratios.index.intersection(valuation_ratios_names)]}

You can either select directly the profitability ratios proposed by the API, by uing this code: `companies.ratios.collect_profitability_ratios()`, or choose specific ones (37 ratios are available for all categories).

The same for valuation ratios: `companies.ratios.collect_valuation_ratios()`

Here is a run example:

In [ ]:
ratios =  get_company_fundamentals("AAPL")

Obtaining historical data: 100%|██████████| 2/2 [00:00<00:00,  9.71it/s]


ratios 2022      NaN
2023    0.275
2024   0.2613
2025   0.3093
Freq: Y-DEC, Name: Return on Assets, dtype: float64


In [ ]:
ratios

{'profitability_ratios':                     2022   2023   2024   2025
 Gross Margin      0.4331 0.4413 0.4621 0.4691
 Operating Margin  0.3029 0.2982 0.3151 0.3197
 Net Profit Margin 0.2531 0.2531 0.2397 0.2692
 Return on Assets     NaN  0.275 0.2613 0.3093
 Return on Equity     NaN 1.7195 1.5741 1.7142,
 'valuation_ratios':                            2022    2023    2024    2025
 Price-to-Earnings        21.254 31.3868 41.1631  36.418
 Price-to-Book           41.8616 48.9873 67.7525 55.3236
 Price-to-Cash-Flow      17.3655 27.5403 32.6289 36.5905
 Price-to-Free-Cash-Flow 19.0341 30.5711 35.4618  41.301
 EV-to-Sales              5.6553   8.188 10.0953  9.9914
 EV-to-EBITDA            17.0831 24.9432 29.3152 28.7259}

In [ ]:
ratios['profitability_ratios']

,2022,2023,2024,2025
Gross Margin,0.4331,0.4413,0.4621,0.4691
Operating Margin,0.3029,0.2982,0.3151,0.3197
Net Profit Margin,0.2531,0.2531,0.2397,0.2692
Return on Assets,NaN,0.275,0.2613,0.3093
Return on Equity,NaN,1.7195,1.5741,1.7142


Now let's define our agentic system:

# Step-by-step: Agentic System: Sequential - Prompt-chaining

## Tools

We'll use the same function than before as a tool to be added to our first agent `fundamental_retrieval_agent`:

In [ ]:
@function_tool
def get_company_fundamentals(ticker: str) -> dict[str:pd.DataFrame]:
  """ Get the profitability and valuation fundamental ratios for a given ticker. """

  companies = Toolkit(
      [ticker], api_key=FINANCIAL_MODELING_PREP_API_KEY, start_date="2022-01-01"
  )
  ratios = companies.ratios.collect_all_ratios()

  # Keep only profitability and valuation ratios
  profitability_ratios_names = ['Return on Equity','Return on Assets','Gross Margin','Operating Margin','Net Profit Margin']
  valuation_ratios_names = ['Price-to-Earnings',  "Price-to-Book", "Price-to-Cash-Flow", "Price-to-Free-Cash-Flow", 'EV-to-EBITDA','EV-to-Sales']

  return {"profitability_ratios":ratios.loc[ratios.index.intersection(profitability_ratios_names)],
   "valuation_ratios":ratios.loc[ratios.index.intersection(valuation_ratios_names)]}

In [ ]:
MODEL = 'gpt-4o-mini'

## Agents

### 1- Fundamental Retrieval Agent

  This agent retrieves fundamental ratios from the Financial Modeling Prep (FMP) API using the tool `get_company_fundamentals` and returns the data as a structured JSON output.

In [ ]:
fundamental_retrieval_agent = Agent(
    name="Fundamental Retrieval Agent",
    model=MODEL,
    instructions="""
You are the Fundamental Retrieval Agent.

Tasks:
1) Call the fundamentals tool for the specified ticker.
2) Output ONLY valid JSON with this schema: Keep the output strictly factual: no interpretation.

{
  "ticker": "...",
  "fundamentals": { ... },      // raw numbers from tools
}

""",
    tools=[get_company_fundamentals],
)

Running the agent:

With: Input query = "Provide Apple fundamentals"

In [ ]:
fundamental_results = await Runner.run(fundamental_retrieval_agent, 'Provide Apple fundamentals')

Obtaining historical data: 100%|██████████| 2/2 [00:00<00:00,  9.65it/s]


ratios 2022      NaN
2023    0.275
2024   0.2613
2025   0.3093
Freq: Y-DEC, Name: Return on Assets, dtype: float64


Results of the *fundamental_retrieval_agent* agent:

In [ ]:
print(fundamental_results.final_output)

```json
{
  "ticker": "AAPL",
  "fundamentals": {
    "profitability_ratios": {
      "2022": {
        "Gross Margin": 0.4331,
        "Operating Margin": 0.3029,
        "Net Profit Margin": 0.2531,
        "Return on Assets": null,
        "Return on Equity": null
      },
      "2023": {
        "Gross Margin": 0.4413,
        "Operating Margin": 0.2982,
        "Net Profit Margin": 0.2531,
        "Return on Assets": 0.275,
        "Return on Equity": 1.7195
      },
      "2024": {
        "Gross Margin": 0.4621,
        "Operating Margin": 0.3151,
        "Net Profit Margin": 0.2397,
        "Return on Assets": 0.2613,
        "Return on Equity": 1.5741
      },
      "2025": {
        "Gross Margin": 0.4691,
        "Operating Margin": 0.3197,
        "Net Profit Margin": 0.2692,
        "Return on Assets": 0.3093,
        "Return on Equity": 1.7142
      }
    },
    "valuation_ratios": {
      "2022": {
        "Price-to-Earnings": 21.254,
        "Price-to-Book": 41.8616,
  

We get the profitability and valuations ratios for several years.

A shown, the ouput of this first agent `fundamental_retrieval_agent` is a structured JSON format.

This output is then used as **input** in the following agent `fundamental_analysis_agent`.

### 2-Fundamental Analysis Agent

  This agent consumes the financial ratios produced by the first agent (`fundamental_results.final_output`) and interprets them from a fundamental perspective.

  Based on these inputs, it evaluates the strength or weakness of key dimensions such as profitability, valuation, and risk, and provides a short justification for each assessment.

In [ ]:
fundamental_analysis_agent = Agent(
    name="Fundamental Analysis Agent",
    model=MODEL,
    instructions="""
You are the Fundamental Analysis Agent.

Input: JSON from the Retrieval Agent.

Tasks:
1) Interpret the fundamentals along 2 dimensions:
   - Profitability (profitability, margins, ROE)
   - Valuation (P/E, E/S)
2) Identify 2-3 key risks (e.g., valuation risk, margin sustainability).
3) Output ONLY valid JSON with this schema:

{
  "ticker": "...",
  "fundamentals": { ... },   // keep from input
  "analysis": {
    "Profitability": { "rating": "strong|neutral|weak", "evidence": [string, ...] },
    "valuation": { "rating": "cheap|fair|expensive", "evidence": [string, ...] },
    "risks": [string, ...]
  }
}

Rules:
- Keep evidence short and specific.
""",
)

This agent is instructed to interpret the fundamentals along two dimensions, profitability and valuation, by assigning a rating (strong|neutral|weak or cheap|fair|expensive) and providing concise evidence to justify each assessment. In addition, it identifies key risks that may arise from the observed ratio levels.

Running the agent `fundamental_analysis_agent`:

In [ ]:
fundamental_analysis = await Runner.run(fundamental_analysis_agent, fundamental_results.final_output)
print(fundamental_analysis.final_output)

```json
{
  "ticker": "AAPL",
  "fundamentals": {
    "profitability_ratios": {
      "2022": {
        "Gross Margin": 0.4331,
        "Operating Margin": 0.3029,
        "Net Profit Margin": 0.2531,
        "Return on Assets": null,
        "Return on Equity": null
      },
      "2023": {
        "Gross Margin": 0.4413,
        "Operating Margin": 0.2982,
        "Net Profit Margin": 0.2531,
        "Return on Assets": 0.275,
        "Return on Equity": 1.7195
      },
      "2024": {
        "Gross Margin": 0.4621,
        "Operating Margin": 0.3151,
        "Net Profit Margin": 0.2397,
        "Return on Assets": 0.2613,
        "Return on Equity": 1.5741
      },
      "2025": {
        "Gross Margin": 0.4691,
        "Operating Margin": 0.3197,
        "Net Profit Margin": 0.2692,
        "Return on Assets": 0.3093,
        "Return on Equity": 1.7142
      }
    },
    "valuation_ratios": {
      "2022": {
        "Price-to-Earnings": 21.254,
        "Price-to-Book": 41.8616,
  

The output from this agent, `fundamental_analysis_agent`, is also a structured JSON format, as requested in the instructions. This will be used as input in the following agent `decision_agent`.

### 3-Decision Agent

This agent is responsible for making the final investment recommendation (Buy, Hold or Sell) based on the analysis produced by the second agent (`fundamental_analysis.final_output`):

In [ ]:
decision_agent = Agent(
    name="Decision Agent",
    model=MODEL,
    instructions="""
You are the Decision Agent.

Input: JSON from the Fundamental Analysis Agent.

Tasks: Based only on the input coming from the Fundamental Analysis Agent:
1) Produce a simple stance: BUY, HOLD, or SELL.
2) Provide 3 bullet reasons grounded in the analysis.
3) Provide a "watchlist" section: what metrics would change your stance.

Output ONLY valid JSON:

{
  "ticker": "...",
  "stance": "BUY|HOLD|SELL",
  "summary": {
    "reasons": [string, string, string],
    "watchlist": [string, string, string]
  }
}

""",
)

Running the "decision_agent":

In [ ]:
final_decision = await Runner.run(decision_agent, fundamental_analysis.final_output)
print(final_decision.final_output)

```json
{
  "ticker": "AAPL",
  "stance": "HOLD",
  "summary": {
    "reasons": [
      "Strong profitability with improving Gross and Operating Margins.",
      "Consistent Net Profit Margins around 25% indicate solid earnings.",
      "While the valuations are high, the company’s fundamental performance remains strong."
    ],
    "watchlist": [
      "Decrease in valuation ratios such as P/E or Price-to-Book.",
      "Sustained Net Profit Margin decline below 25%.",
      "Market developments that could affect growth prospects."
    ]
  }
}
```


This final output is following the structured format specified in the instructions.

It results in a HOLD recommendation, reflecting a balance between strong profitability and high valuation levels. Although the agent highlights solid margins and good earnings performance, elevated valuation ratios prevent a more positive recommendation.
The watchlist elements provide actionable signals, indicating which changes in valuation or profitability metrics could justify a future revision of the recommendation.

This structured output demonstrates how the agentic system not only delivers a decision but also makes the underlying rationale and decision boundaries explicit.


# All in one place: Agentic System

In [ ]:
fundamental_retrieval_agent = Agent(
    name="Fundamental Retrieval Agent",
    model=MODEL,
    instructions="""
You are the Fundamental Retrieval Agent.

Tasks:
1) Call the fundamentals tool for the ticker.
2) Output ONLY valid JSON with this schema: Keep the output strictly factual: no interpretation.

{
  "ticker": "...",
  "fundamentals": { ... },      // raw numbers from tools
}

""",
    tools=[get_company_fundamentals],
)

fundamental_analysis_agent = Agent(
    name="Fundamental Analysis Agent",
    model=MODEL,
    instructions="""
You are the Fundamental Analysis Agent.

Input: JSON from the Retrieval Agent.

Tasks:
1) Interpret the fundamentals along 2 dimensions:
   - Profitability (profitability, margins, ROE)
   - Valuation (P/E, E/S)
2) Identify 2-3 key risks (e.g., valuation risk, margin sustainability).
3) Output ONLY valid JSON with this schema:

{
  "ticker": "...",
  "fundamentals": { ... },   // keep from input
  "analysis": {
    "Profitability": { "rating": "strong|neutral|weak", "evidence": [string, ...] },
    "valuation": { "rating": "cheap|fair|expensive", "evidence": [string, ...] },
    "risks": [string, ...]
  }
}

Rules:
- Base judgments only on provided numbers and peer table.
- Keep evidence short and specific.
""",
)

decision_agent = Agent(
    name="Decision Agent",
    model=MODEL,
    instructions="""
You are the Decision Agent.

Input: JSON from the Fundamental Analysis Agent.

Tasks: Based only on the input coming from the Fundamental Analysis Agent:
1) Produce a simple stance: BUY, HOLD, or SELL.
2) Provide 3 bullet reasons grounded in the analysis.
3) Provide a "watchlist" section: what metrics would change your stance.

Output ONLY valid JSON:

{
  "ticker": "...",
  "stance": "BUY|HOLD|SELL",
  "summary": {
    "reasons": [string, string, string],
    "watchlist": [string, string, string]
  }
}

""",
)

In [ ]:
# Step 1: Retrieve fundamental data
fundamental_results = await Runner.run(
    fundamental_retrieval_agent,
    'Provide Apple fundamentals'
)
print(fundamental_results.final_output)

# Step 2: Analyze fundamentals
fundamental_analysis = await Runner.run(
    fundamental_analysis_agent,
    fundamental_results.final_output
)
print(fundamental_analysis.final_output)

# Step 3: Generate investment decision
final_decision = await Runner.run(
    decision_agent,
    fundamental_analysis.final_output
)
print(final_decision.final_output)